# 확률분포와 손실함수: MLE에서 유도하기

[Step 6 이슈 #9](https://github.com/ntt65/A2_1/issues/9). 수학 계산은 NumPy, 그림은 Matplotlib로 수행한다.
정규분포 표기는 **$N(\mu,\sigma^2)$: 평균과 분산**이다. 모든 로그는 자연로그이며 손실의 **합과 평균**을 구분한다.

정규·베르누이 분포의 질량을 수치로 확인한 뒤 Gaussian, Bernoulli, categorical likelihood에서 각 손실을 유도한다.
새 커널에서 위에서 아래로 실행한다. 독립 고정값·해석식과 명시적 허용오차를 assertion으로 검증하며, 확률을 임의로 clipping하지 않는다.

In [1]:
import json
import os
from pathlib import Path
import sys

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "utils" / "math.py").is_file():
    candidate = repo_root.parent
    if repo_root.name == "notebooks" and (candidate / "src" / "utils" / "math.py").is_file():
        repo_root = candidate
    else:
        raise RuntimeError("Run from the A2_1 root or its notebooks directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils.math import sigmoid, softmax, _as_finite_real_array
from src.utils.numerical import _finite_scalar
from src.utils.random import set_seed
from src.calculus import numerical_gradient

set_seed(42)
output_dir = Path(os.environ.get("A2_1_OUTPUT_DIR", str(repo_root / "outputs"))).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
print(f"NumPy {np.__version__}; float64; seed=42; deterministic grids")

NumPy 2.2.6; float64; seed=42; deterministic grids


## 1. 정규분포 PDF

$$p(x\mid\mu,\sigma^2)=\frac{1}{\sqrt{2\pi}\sigma}
\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right),\qquad \sigma^2>0.$$

`N(0,1)`의 표준편차는1, `N(2,0.5)`의 표준편차는 $\sqrt{0.5}$다. 분산0.5를 표준편차0.5로 해석하지 않는다.
PDF는 확률 **밀도**이므로 개별 높이가1보다 클 수 있으며, 전체 적분이1이다.
각 분포의 $\mu\pm8\sigma$ 구간을10,001점으로 나누어 사다리꼴 적분한다. 유한 구간의 꼬리 누락과 수치 적분 오차가 있으므로 정확히1이라고 강제하지 않고 절대오차 $\le10^{-4}$를 검사한다.

In [2]:
def normal_pdf(x, mu, variance):
    """Return the normal density with x's shape, scalar mean and positive variance.

    Inputs are finite real float64 values; mu and variance have shape ().
    Non-real input raises TypeError; invalid scalar parameters or variance
    raises ValueError. Extreme finite tails can underflow to zero, which is
    the float64 density limit. The input is never changed.
    """
    values = _as_finite_real_array(x, "x")
    location = _finite_scalar(mu, "mu")
    spread = _finite_scalar(variance, "variance")
    if spread <= 0:
        raise ValueError("variance must be positive")
    sigma = np.sqrt(spread)
    with np.errstate(over="ignore", under="ignore"):
        standardized = (values-location)/sigma
        density = np.exp(-0.5*standardized**2)/(np.sqrt(2*np.pi)*sigma)
    if not np.all(np.isfinite(density)):
        raise ValueError("density must be representable as finite float64")
    return np.asarray(density)


normals = []
for mu, variance, expected_peak in ((0.0, 1.0, 0.3989422804014327),
                                   (2.0, 0.5, 0.5641895835477563)):
    sigma = np.sqrt(variance)
    grid = np.linspace(mu-8*sigma, mu+8*sigma, 10001)
    density = normal_pdf(grid, mu, variance)
    integral = float(np.trapezoid(density, grid))
    peak = float(normal_pdf(mu, mu, variance))
    assert density.shape == (10001,) and np.all(density >= 0)
    assert abs(integral-1.0) <= 1e-4
    assert abs(peak-expected_peak) <= 1e-15
    np.testing.assert_allclose(normal_pdf([mu-sigma, mu+sigma], mu, variance),
                               np.full(2, expected_peak*np.exp(-0.5)), atol=1e-15, rtol=0)
    normals.append({"mu": mu, "variance": variance, "sigma": float(sigma),
                    "grid_points": len(grid), "integral": integral,
                    "integral_absolute_error": abs(integral-1.0), "peak": peak,
                    "minimum_pdf": float(np.min(density))})
for invalid in (0.0, -1.0, np.inf, np.nan):
    try:
        normal_pdf([0.0], 0.0, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid variance was accepted")
for item in normals:
    print(item)

figure, axes = plt.subplots(figsize=(8.2, 4.8), layout="constrained")
try:
    display_x = np.linspace(-5.0, 6.0, 1001)
    for item in normals:
        axes.plot(display_x, normal_pdf(display_x, item["mu"], item["variance"]),
                  label=f"N(mu={item['mu']:g}, variance={item['variance']:g})")
    axes.set(xlabel="x", ylabel="Probability density", title="Normal densities: second parameter is variance")
    axes.legend()
    axes.grid(alpha=0.25)
    figure.savefig(output_dir / "probability_normal.png", dpi=140)
finally:
    plt.close(figure)

{'mu': 0.0, 'variance': 1.0, 'sigma': 1.0, 'grid_points': 10001, 'integral': 0.9999999999999989, 'integral_absolute_error': 1.1102230246251565e-15, 'peak': 0.3989422804014327, 'minimum_pdf': 5.052271083536893e-15}
{'mu': 2.0, 'variance': 0.5, 'sigma': 0.7071067811865476, 'grid_points': 10001, 'integral': 0.999999999999999, 'integral_absolute_error': 9.992007221626409e-16, 'peak': 0.5641895835477563, 'minimum_pdf': 7.144990287123286e-15}


![평균과 분산이 표시된 정규분포](../outputs/probability_normal.png)

폭과 높이의 차이를 확인한다. 높은 PDF 값 자체는 개별 점의 확률이 아니다.

## 2. 베르누이 PMF

$$P(Y=y)=p^y(1-p)^{1-y},\qquad y\in\{0,1\},\quad 0\le p\le1.$$

support는0과1이며 확률벡터는 `[1-p,p]`다. `p=.3`은 `[.7,.3]`, `p=.7`은 `[.3,.7]`이고 합은1이다.
연속 밀도와 달리 PMF는 각 support의 확률 자체다.

In [3]:
def bernoulli_pmf(p):
    """Return probabilities of support [0,1], shape (2,), for scalar p in [0,1].

    Finite real scalar input is required. Bad type raises TypeError and bad
    shape, nonfinite p, or a probability outside [0,1] raises ValueError.
    """
    probability = _finite_scalar(p, "p")
    if not 0 <= probability <= 1:
        raise ValueError("p must be in [0,1]")
    return np.array([1-probability, probability], dtype=np.float64)


bernoulli = []
for p, reference in ((0.3, [0.7, 0.3]), (0.7, [0.3, 0.7])):
    pmf = bernoulli_pmf(p)
    np.testing.assert_allclose(pmf, reference, atol=1e-15, rtol=0)
    assert np.all((pmf >= 0) & (pmf <= 1))
    mass_error = abs(float(np.sum(pmf))-1.0)
    assert mass_error <= 1e-12
    bernoulli.append({"p": p, "support": [0,1], "pmf": pmf.tolist(), "mass_error": mass_error})
np.testing.assert_array_equal(bernoulli_pmf(0), [1,0])
np.testing.assert_array_equal(bernoulli_pmf(1), [0,1])
for invalid in (-0.1, 1.1, np.nan):
    try:
        bernoulli_pmf(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid Bernoulli probability was accepted")
print(bernoulli)

figure, axes = plt.subplots(figsize=(7.2, 4.6), layout="constrained")
try:
    support = np.array([0,1])
    for shift, item in zip((-0.18, 0.18), bernoulli):
        bars = axes.bar(support+shift, item["pmf"], width=0.34, label=f"p={item['p']:g}")
        axes.bar_label(bars, fmt="%.1f", padding=3)
    axes.set(xticks=support, xlabel="Outcome y", ylabel="Probability mass",
             ylim=(0,1), title="Bernoulli probability mass on {0,1}")
    axes.legend()
    axes.grid(axis="y", alpha=0.2)
    figure.savefig(output_dir / "probability_bernoulli.png", dpi=140)
finally:
    plt.close(figure)

[{'p': 0.3, 'support': [0, 1], 'pmf': [0.7, 0.3], 'mass_error': 0.0}, {'p': 0.7, 'support': [0, 1], 'pmf': [0.30000000000000004, 0.7], 'mass_error': 0.0}]


![베르누이 PMF 비교](../outputs/probability_bernoulli.png)

## 3. Softmax: 같은 상수를 빼도 확률은 같다

$$p_j=\frac{e^{z_j}}{\sum_k e^{z_k}}
=\frac{e^{z_j-m}}{\sum_k e^{z_k-m}},\qquad m=\max_k z_k.$$

공통 모듈을 재사용한다. 아래 `(2,3)` 배열은 행마다 하나의 범주 분포이며 `axis=-1`로 정규화한다.
다른 축을 지정하면 정규화되는 분포 묶음도 바뀐다. 유한한 큰 logit에서도 확률합 오차 $\le10^{-6}$인지 확인한다.

In [4]:
logits = np.array([[1000.0, 1001.0, 1002.0], [-1000.0, 0.0, 1000.0]])
probabilities = softmax(logits, axis=-1)
row_sums = np.sum(probabilities, axis=-1)
assert probabilities.shape == logits.shape
assert np.all(np.isfinite(probabilities)) and np.all(probabilities >= 0)
max_sum_error = float(np.max(np.abs(row_sums-1)))
assert max_sum_error <= 1e-6
np.testing.assert_allclose(probabilities[0], [0.09003057317038046, 0.24472847105479764, 0.6652409557748218], atol=1e-15, rtol=0)
np.testing.assert_allclose(softmax(np.log([2.0,3.0,5.0])), [0.2,0.3,0.5], atol=1e-15, rtol=0)
column_probabilities = softmax(logits, axis=0)
column_sum_error = float(np.max(np.abs(column_probabilities.sum(axis=0)-1)))
assert column_sum_error <= 1e-6
softmax_report = {"logits": logits.tolist(), "probabilities": probabilities.tolist(),
                  "axis": -1, "row_sums": row_sums.tolist(),
                  "max_sum_error": max_sum_error, "column_sum_error": column_sum_error}
print(softmax_report)

{'logits': [[1000.0, 1001.0, 1002.0], [-1000.0, 0.0, 1000.0]], 'probabilities': [[0.09003057317038046, 0.24472847105479764, 0.6652409557748218], [0.0, 0.0, 1.0]], 'axis': -1, 'row_sums': [0.9999999999999999, 1.0], 'max_sum_error': 1.1102230246251565e-16, 'column_sum_error': 1.1102230246251565e-16}


## 4. Gaussian MLE가 MSE 최소화와 연결되는 조건

관측 $y_i=f_\theta(x_i)+\epsilon_i$에서 오차는 서로 독립이며 $\epsilon_i\sim N(0,\sigma^2)$라고 가정한다.
모든 표본에 같은 양의 분산 $\sigma^2$를 사용하며, 여기서는 그 분산을 **알려진 상수**로 고정한다.

$$\mathcal L(\theta)=\prod_{i=1}^n\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left[-\frac{(y_i-f_\theta(x_i))^2}{2\sigma^2}\right].$$

곱의 자연로그를 합으로 바꾸고 부호를 반전하면

$$\operatorname{NLL}(\theta)
=\frac n2\ln(2\pi\sigma^2)+\frac{1}{2\sigma^2}\sum_{i=1}^n(y_i-f_\theta(x_i))^2
=\frac n2\ln(2\pi\sigma^2)+\frac{n}{2\sigma^2}\operatorname{MSE}(\theta).$$

첫 항은 $\theta$와 무관하고 MSE의 계수는 양수이므로 같은 허용 매개변수 집합에서 MLE, SSE 최소화, MSE 최소화가 **같은 최적해 집합**을 갖는다.
NLL과 MSE가 항상 같은 숫자라는 뜻은 아니다. 평균 NLL은 위 식을 n으로 나눈 값이며, 양의 상수 배율은 최적해를 바꾸지 않는다.

분산이 입력 또는 $\theta$에 따라 달라지면 로그 분산 항과 오차별 가중치를 버릴 수 없다. 분산까지 추정하는 문제는 위 고정분산 유도와 구분해야 한다.
연속 확률의 밀도는1보다 클 수 있으므로 Gaussian NLL이 음수일 수도 있다. 일반적인 비음수 assertion을 두지 않는다.

In [5]:
observed = np.array([0.0, 1.0, 2.0])
predicted = np.array([1.0, 0.5, 1.5])
variance = 0.5
n = observed.size
sse = float(np.sum((observed-predicted)**2))
mse = sse/n
gaussian_nll = n/2*np.log(2*np.pi*variance)+sse/(2*variance)
independent_gaussian_reference = 3.21709482877410026
assert sse == 1.5 and mse == 0.5
assert abs(gaussian_nll-independent_gaussian_reference) <= 1e-14
direct_nll = -float(np.sum([np.log(normal_pdf(y, mean, variance)) for y, mean in zip(observed, predicted)]))
identity_error = abs(gaussian_nll-direct_nll)
assert identity_error <= 1e-14
assert gaussian_nll != mse
assert 0.5*np.log(2*np.pi*0.01) < 0  # Zero residual at a narrow density's peak.
gaussian_report = {"n": int(n), "variance": variance, "sse": sse, "mse": mse,
                   "nll": float(gaussian_nll), "independent_reference": independent_gaussian_reference,
                   "identity_error": float(identity_error), "mean_nll": float(gaussian_nll/n)}
print(gaussian_report)

{'n': 3, 'variance': 0.5, 'sse': 1.5, 'mse': 0.5, 'nll': 3.2170948287741004, 'independent_reference': 3.2170948287741004, 'identity_error': 4.440892098500626e-16, 'mean_nll': 1.0723649429247002}


## 5. Bernoulli likelihood에서 BCE 유도

$y_i\in\{0,1\}$, $p_i=\sigma(z_i)$인 독립 표본의 likelihood는
$$\mathcal L=\prod_i p_i^{y_i}(1-p_i)^{1-y_i}.$$
따라서 음의 로그우도는
$$L_{\mathrm{sum}}=-\sum_i\left[y_i\ln p_i+(1-y_i)\ln(1-p_i)\right],\qquad
L_{\mathrm{mean}}=L_{\mathrm{sum}}/n.$$

한 표본에서 $p=\sigma(z)$를 대입하면
$$\ell(z,y)=\log(1+e^z)-yz=\operatorname{logaddexp}(0,z)-yz,\qquad
\frac{\partial\ell}{\partial z}=\sigma(z)-y.$$
sum 손실의 각 logit 미분은 위 식이고 mean 손실의 각 미분은 n으로 나눈다. 앞 역전파 노트북은 단일 표본이어서 평균 계수가 없었다.

`logaddexp`는 exp의 overflow를 피하지만 큰 양수 z와 y=1에서는 마지막 뺄셈이 작은 양의 손실을0으로 상쇄할 수 있다.
아래 binary-label 계산은 수학적으로 같은 $\operatorname{logaddexp}(0,(1-2y)z)$를 사용해 그 작은 값을 보존한다.
미분식도 float64에서 sigmoid가 포화되면 작은 꼬리값을 표현하지 못할 수 있으므로, 충분히 중간인 logit에서 중심차분과 비교한다.

**경계 확률:** $0\log0=0$은 해당 항의 연속 극한이다. p=0,y=0 또는 p=1,y=1의 손실은0이다.
불가능한 사건(p=0,y=1 또는 p=1,y=0)은 손실 $+\infty$다. `0*log(0)`으로 NaN을 만든 뒤 숨기거나 임의 clipping으로 다른 손실을 만들지 않는다.

In [6]:
def _binary_pair(y, values, name):
    """Return finite real arrays with identical nonempty shape and binary labels.

    Scalars have shape (). Invalid type raises TypeError; mismatched/empty
    shape, nonfinite values or labels outside {0,1} raise ValueError.
    """
    labels = _as_finite_real_array(y, "y")
    data = _as_finite_real_array(values, name)
    if labels.shape != data.shape or labels.size == 0:
        raise ValueError("labels and values must have identical nonempty shapes")
    if not np.all((labels == 0) | (labels == 1)):
        raise ValueError("binary labels must be 0 or 1")
    return labels, data


def probability_bce(y, p):
    """Return elementwise binary NLL with y/p's shape, including valid +inf boundaries.

    Labels are binary and p lies in [0,1]. Select the observed-event probability
    before taking its log, avoiding 0*log(0). Impossible observations produce
    +inf deliberately; no clipping is performed. Invalid input raises.
    """
    labels, probabilities = _binary_pair(y, p, "p")
    if np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("p must be in [0,1]")
    observed_probability = np.where(labels == 1, probabilities, 1-probabilities)
    with np.errstate(divide="ignore"):
        return -np.log(observed_probability)


def binary_logit_loss(y, z):
    """Return elementwise binary logit NLL with matching nonempty y/z shape.

    For binary labels, logaddexp(0,(1-2y)*z) equals logaddexp(0,z)-y*z
    while retaining small positive losses for large correct logits. Inputs
    must be finite real with binary labels. Invalid inputs raise; tails may
    still underflow to zero when below float64's range.
    """
    labels, logits = _binary_pair(y, z, "z")
    with np.errstate(under="ignore"):
        return np.logaddexp(0.0, (1-2*labels)*logits)

In [7]:
binary_y = np.array([0.0,1.0])
binary_p = np.array([0.3,0.7])
binary_z = np.log(binary_p/(1-binary_p))
probability_losses = probability_bce(binary_y, binary_p)
logit_losses = binary_logit_loss(binary_y, binary_z)
bce_sum = float(np.sum(probability_losses))
bce_mean = float(np.mean(probability_losses))
assert abs(bce_sum-0.71334988787746476) <= 1e-15
np.testing.assert_allclose(logit_losses, probability_losses, atol=1e-15, rtol=0)
np.testing.assert_allclose(logit_losses, np.logaddexp(0,binary_z)-binary_y*binary_z, atol=1e-15, rtol=0)
assert abs(bce_mean-bce_sum/2) <= 1e-15
analytic_sum_gradient = sigmoid(binary_z)-binary_y
numeric_sum_gradient = numerical_gradient(lambda z: float(np.sum(binary_logit_loss(binary_y,z))), binary_z)
numeric_mean_gradient = numerical_gradient(lambda z: float(np.mean(binary_logit_loss(binary_y,z))), binary_z)
np.testing.assert_allclose(numeric_sum_gradient, [0.3,-0.3], atol=1e-8, rtol=0)
np.testing.assert_allclose(numeric_sum_gradient, analytic_sum_gradient, atol=1e-8, rtol=0)
np.testing.assert_allclose(numeric_mean_gradient, analytic_sum_gradient/2, atol=1e-8, rtol=0)
max_gradient_error = float(np.max(np.abs(numeric_sum_gradient-analytic_sum_gradient)))

boundary_losses = probability_bce([0,1,1,0], [0,1,0,1])
np.testing.assert_array_equal(boundary_losses[:2], [0,0])
assert np.all(np.isposinf(boundary_losses[2:])) and not np.any(np.isnan(boundary_losses))
extreme_losses = binary_logit_loss([0,1,1,0], [-1000,1000,-1000,1000])
np.testing.assert_allclose(extreme_losses, [0,0,1000,1000], atol=0, rtol=0)
assert np.all(np.isfinite(extreme_losses))
positive_tail_at_40 = float(binary_logit_loss(1,40))
np.testing.assert_allclose(positive_tail_at_40, 4.248354255291589e-18, atol=0, rtol=1e-14)
assert positive_tail_at_40 > 0
cancellation_example = float(np.logaddexp(0.0,40.0)-40.0)
for invalid_y, invalid_p in (([0.5],[0.3]), ([1],[1.1]), ([0,1],[0.2])):
    try:
        probability_bce(invalid_y, invalid_p)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid binary input was accepted")
bce_report = {"y": binary_y.tolist(), "probabilities": binary_p.tolist(), "sum_loss": bce_sum,
              "mean_loss": bce_mean, "logits": binary_z.tolist(),
              "logit_sum_loss": float(np.sum(logit_losses)), "max_gradient_error": max_gradient_error,
              "analytic_sum_gradient": analytic_sum_gradient.tolist(), "numerical_sum_gradient": numeric_sum_gradient.tolist(),
              "extreme_losses": extreme_losses.tolist(), "positive_tail_at_40": positive_tail_at_40,
              "subtractive_formula_at_40": cancellation_example, "boundary_policy_verified": True}
print(bce_report)

{'y': [0.0, 1.0], 'probabilities': [0.3, 0.7], 'sum_loss': 0.7133498878774649, 'mean_loss': 0.35667494393873245, 'logits': [-0.8472978603872036, 0.8472978603872034], 'logit_sum_loss': 0.7133498878774649, 'max_gradient_error': 8.626654945942391e-12, 'analytic_sum_gradient': [0.3, -0.30000000000000004], 'numerical_sum_gradient': [0.3000000000030756, -0.3000000000086267], 'extreme_losses': [0.0, 0.0, 1000.0, 1000.0], 'positive_tail_at_40': 4.248354255291589e-18, 'subtractive_formula_at_40': 0.0, 'boundary_policy_verified': True}


## 6. 다중 클래스 likelihood에서 Cross-Entropy 유도

각 표본의 one-hot 벡터 $y_{ij}$는 실제 클래스 하나에서만1이며 $\sum_j y_{ij}=1$이다.
$p_{ij}=\operatorname{softmax}(z_i)_j$이면 독립 표본 likelihood와 음의 로그는

$$\mathcal L=\prod_i\prod_j p_{ij}^{y_{ij}},\qquad
L_{\mathrm{sum}}=-\sum_i\sum_j y_{ij}\ln p_{ij}=-\sum_i\ln p_{i,c_i}.$$

이는 categorical Cross-Entropy이고, 표본 평균은 n으로 나눈다. 이진 Bernoulli의 두 결과 항에서 나온 BCE와 유도 출발점을 구분한다.
target가0인 클래스의 $0\log0$ 항은0이고 실제 클래스의 확률이0이면 손실은 $+\infty$다.

logit에서 직접 계산할 때 $s=z-\max(z)$로 이동시켜
$$\ell(z,c)=\ln\sum_j e^{s_j}-s_c$$
를 사용한다. 큰 공통 offset을 더한 logsumexp와 target logit을 마지막에 빼는 취소 오차도 줄인다.
한 표본의 logit 미분은 $p-y_{\mathrm{onehot}}$이며, 평균 손실이면 n으로 나눈다.

In [8]:
def _class_vector(values, target, name):
    """Validate a nonempty real finite class vector (K,) and integer target in [0,K).

    Invalid numeric/index types raise TypeError; bad dimensions, empty vectors,
    nonfinite values, and out-of-range indices raise ValueError.
    """
    vector = _as_finite_real_array(values, name)
    if vector.ndim != 1 or vector.size == 0:
        raise ValueError("class values must be a nonempty vector")
    if isinstance(target,(bool,np.bool_)) or not isinstance(target,(int,np.integer)):
        raise TypeError("target must be an integer class index")
    if not 0 <= target < vector.size:
        raise ValueError("target index is out of range")
    return vector, int(target)


def categorical_probability_ce(probabilities, target):
    """Return scalar observed-class NLL for a normalized probability vector (K,).

    Probabilities lie in [0,1] and sum to one within absolute tolerance1e-12.
    A zero target probability returns +inf; zero non-target probabilities
    contribute nothing. No clipping occurs. Invalid inputs raise.
    """
    vector, index = _class_vector(probabilities,target,"probabilities")
    if np.any((vector < 0) | (vector > 1)) or abs(float(vector.sum())-1) > 1e-12:
        raise ValueError("probabilities must be nonnegative and sum to one")
    with np.errstate(divide="ignore"):
        return float(-np.log(vector[index]))


def categorical_logit_ce(logits, target):
    """Return scalar CE from a finite logit vector (K,) using a common shift.

    log(sum(exp(z-max(z))))-(z_target-max(z)) avoids exponential overflow
    and cancellation of a large common offset. An unrepresentable shifted
    difference/result raises ValueError; invalid vector/index input raises.
    """
    vector, index = _class_vector(logits,target,"logits")
    with np.errstate(over="ignore", under="ignore"):
        shifted = vector-np.max(vector)
        value = np.log(np.sum(np.exp(shifted)))-shifted[index]
    if not np.all(np.isfinite(shifted)) or not np.isfinite(value):
        raise ValueError("shifted logits and CE must be representable")
    return float(value)


class_probabilities = np.array([0.2,0.3,0.5])
target_class = 2
probability_ce = categorical_probability_ce(class_probabilities,target_class)
logit_ce = categorical_logit_ce(np.log([2.0,3.0,5.0]),target_class)
assert abs(probability_ce-0.69314718055994531) <= 1e-15
assert abs(logit_ce-probability_ce) <= 1e-15
extreme_ce = categorical_logit_ce([-1000,0,1000],0)
assert extreme_ce == 2000.0
assert abs(categorical_logit_ce([1e20,1e20,1e20],1)-np.log(3)) <= 1e-15
assert categorical_probability_ce([0,0,1],2) == 0
assert np.isposinf(categorical_probability_ce([0,0,1],0))
for invalid, target in (([0.2,0.3],0), ([-0.2,1.2],0), ([0.2,0.3,0.5],3)):
    try:
        categorical_probability_ce(invalid,target)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid categorical input was accepted")
categorical_report = {"probabilities": class_probabilities.tolist(), "target": target_class,
                      "probability_ce": probability_ce, "logit_ce": logit_ce,
                      "extreme_ce": extreme_ce, "boundary_policy_verified": True}
print(categorical_report)

{'probabilities': [0.2, 0.3, 0.5], 'target': 2, 'probability_ce': 0.6931471805599453, 'logit_ce': 0.6931471805599453, 'extreme_ce': 2000.0, 'boundary_policy_verified': True}


## 7. 확인한 연결과 범위

- Gaussian NLL과 MSE의 최적해 동치는 독립 오차·고정 양의 공통 분산 가정에서 성립한다.
- Bernoulli의 음의 로그가 BCE, categorical의 음의 로그가 one-hot Cross-Entropy다.
- 자연로그·손실합/평균·gradient의 n계수를 구분했다.
- 경계 확률의 불가능 사건은 실제로 무한 손실이며 clipping으로 숨기지 않았다. JSON에는 무한 수를 쓰지 않고 경계 검증 여부를 기록한다.
- 수치 안정화도 float64의 표현 한계를 없애지는 않는다. 극소 밀도/확률은0으로 underflow할 수 있다.
- 미분과 손실 검증은 명시한 예제에 대한 것이며 모든 모델이나 데이터의 통계 가정을 보장하지 않는다.

In [9]:
report = {"seed":42, "dtype":"float64", "numpy_version":np.__version__,
          "all_assertions_passed":True, "normals":normals, "bernoulli":bernoulli,
          "softmax":softmax_report, "gaussian":gaussian_report,
          "bce":bce_report, "categorical":categorical_report,
          "figures":["probability_normal.png","probability_bernoulli.png"],
          "notes":["Normal second parameter is variance, not standard deviation.",
                   "Natural logarithms; sum and mean losses are distinguished.",
                   "Gaussian NLL and MSE share minimizers only under the stated fixed-variance assumptions.",
                   "Probability boundaries are not clipped; impossible-event losses are positive infinity.",
                   "Only finite measurements are serialized; boundary assertions are recorded as booleans."]}
(output_dir / "probability_metrics.json").write_text(
    json.dumps(report,ensure_ascii=False,indent=2,allow_nan=False)+"\n",encoding="utf-8",newline="\n")
print("PASS: distributions, normalization, MLE identities, boundary policies, and logit gradients")
print("Saved probability_normal.png, probability_bernoulli.png, probability_metrics.json")

PASS: distributions, normalization, MLE identities, boundary policies, and logit gradients
Saved probability_normal.png, probability_bernoulli.png, probability_metrics.json
